# Ngày 11 · Mô hình sinh: Gaussian Naive Bayes

- Ước lượng tiên nghiệm, trung bình, phương sai từng lớp bằng MLE
- Tính log hậu nghiệm đã chuẩn hóa bằng log-sum-exp
- So mô hình sinh với logistic khi ít dữ liệu train

**Đọc:**
- Stanford CS229, §4.1 Gaussian discriminant analysis; §4.2 Naive bayes
- Vũ Hữu Tiệp, Machine Learning cơ bản, Chương 11 Bộ phân loại naive Bayes; Chương 4 (MLE, MAP)
- ISLR, §4.4 Linear Discriminant Analysis

In [ ]:
import json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

DATA = next((p for p in (Path('data'), Path('../data')) if p.is_dir()), Path('data'))
# Breast Cancer Wisconsin (Diagnostic), UCI, có sẵn trong scikit-learn. 569 mẫu, 30 đặc trưng.
_bc = load_breast_cancer()
X_bc = _bc.data
y_bc = 1 - _bc.target   # 1 = ác tính (lớp cần phát hiện), 0 = lành tính
feature_names = list(_bc.feature_names)

def _check(name, fn):
    try:
        ok, detail = fn()
    except NotImplementedError:
        ok, detail = False, 'Chưa cài đặt (NotImplementedError)'
    except Exception as error:
        ok, detail = False, f'{type(error).__name__}: {error}'
    return {'name': name, 'ok': bool(ok), 'detail': str(detail)}

def _report(ex_id, *checks):
    print('__PY30__' + json.dumps({'id': ex_id, 'checks': list(checks)}, ensure_ascii=False))

# Một phép chia dùng chung cho cả tuần: 70% train, 30% validation, phân tầng theo nhãn.
idx_tr, idx_va = train_test_split(np.arange(len(y_bc)), test_size=0.3, random_state=0, stratify=y_bc)
_scaler = StandardScaler().fit(X_bc[idx_tr])
X_tr, X_va = _scaler.transform(X_bc[idx_tr]), _scaler.transform(X_bc[idx_va])
y_tr, y_va = y_bc[idx_tr], y_bc[idx_va]
from scipy.special import logsumexp
from sklearn.naive_bayes import GaussianNB
_ref11 = GaussianNB().fit(X_tr, y_tr)


## Gaussian Naive Bayes từ đầu

Gợi ý: var = X[y==c].var(axis=0) + 1e-9 * X.var(axis=0).max() (giống scikit-learn). Log mật độ Gauss: −½log(2πσ²) − (x−μ)²/(2σ²).

In [ ]:
def fit_gnb(X, y):
    # Trả dict: classes, prior (C,), mean (C, d), var (C, d).
    raise NotImplementedError

def predict_log_proba_gnb(model, X):
    # Log hậu nghiệm đã chuẩn hóa, shape (n, C).
    raise NotImplementedError


In [ ]:
_report('d11-gnb',
    _check('Tham số khớp scikit-learn', lambda: ((lambda m: np.allclose(m['mean'], _ref11.theta_) and np.allclose(m['var'], _ref11.var_) and np.allclose(m['prior'], _ref11.class_prior_))(fit_gnb(X_tr, y_tr)), 'theta_, var_, class_prior_.')),
    _check('Log hậu nghiệm khớp predict_log_proba', lambda: (np.allclose(predict_log_proba_gnb(fit_gnb(X_tr, y_tr), X_va), _ref11.predict_log_proba(X_va), atol=1e-6), 'Trên tập validation.')),
)


## Dự án

Trên Titanic, so GaussianNB và LogisticRegression khi chỉ dùng 10%, 30%, 100% dữ liệu train (cùng tập validation). Vẽ accuracy theo kích thước train.